# 03 — Feature Engineering & Cleaning

This notebook turns the raw data into a **model-ready dataset** and persists it to
`data/processed/`. Decisions here directly affect model quality, so each step is
justified, not guessed.

Steps:
1. Fix impossible values (`age == 0`)
2. Handle sentinel codes (96/98 → missing)
3. Decide the imputation strategy for `MonthlyIncome` & `NumberOfDependents`
4. Winsorize extreme outliers (utilization, debt ratio)
5. Engineer credit-meaningful features
6. Drop leakage / zero-variance columns
7. Save `train_features.csv` / `test_features.csv` (held-out split)

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
DATA_PATH = "../data/raw/cs-training.csv"
OUT_DIR = "../data/processed"

target = "SeriousDlqin2yrs"
df = pd.read_csv(DATA_PATH, index_col=0)
print(f"Raw: {df.shape[0]:,} rows x {df.shape[1]} columns")

Raw: 150,000 rows x 11 columns


## 1. Fix impossible age

One borrower reports `age == 0`. Age must be positive; we treat it as missing and
impute with the median (age is well-behaved after this).

In [2]:
print("age == 0 count:", (df["age"] == 0).sum())
df.loc[df["age"] < 18, "age"] = np.nan   # also guard early-teen noise
df["age"] = df["age"].fillna(df["age"].median())
print("age after fix -> min:", df["age"].min(), "missing:", df["age"].isnull().sum())

age == 0 count: 1
age after fix -> min: 21.0 missing: 0


## 2. Handle sentinel codes (96 / 98)

The three past-due columns use **96 and 98 as codes**, not real counts. Treating them
as numbers would badly distort the model (98 late payments!). We map them to **missing**
and, because a borrower with a 96/98 code is a *known* high-risk group, we also keep a
**flag** so the model sees that information.

We then impute the (now-missing) counts with **0** — the base-rate assumption: absent
evidence of delinquency, assume none. This is conservative and standard for these columns.

In [3]:
delinq_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

# 1) flag rows that had a sentinel code
sentinel_flag = (df[delinq_cols] >= 96).any(axis=1)
df["had_sentinel_delinq"] = sentinel_flag.astype(int)
print("Rows with a 96/98 code:", sentinel_flag.sum())

# 2) map sentinels to missing, then impute with 0
df[delinq_cols] = df[delinq_cols].where(df[delinq_cols] < 96)
print("max after sentinel fix:", {c: int(df[c].max()) for c in delinq_cols})

df[delinq_cols] = df[delinq_cols].fillna(0)
df[delinq_cols].describe().T[["min", "max", "mean"]]

Rows with a 96/98 code: 269
max after sentinel fix: {'NumberOfTime30-59DaysPastDueNotWorse': 13, 'NumberOfTime60-89DaysPastDueNotWorse': 11, 'NumberOfTimes90DaysLate': 17}


,min,max,mean
NumberOfTime30-59DaysPastDueNotWorse,0.0000,13.0000,0.2454
NumberOfTime60-89DaysPastDueNotWorse,0.0000,11.0000,0.0647
NumberOfTimes90DaysLate,0.0000,17.0000,0.0903


## 3. Imputation strategy

- **`MonthlyIncome` (~20% missing).** Rather than the mean, use the **median** (income is
  heavily right-skewed) computed on the training set only later; here we add a
  **`income_missing` flag** first, then impute. We keep the flag because nb02 showed
  missingness is informative.
- **`NumberOfDependents` (~2.6% missing).** Median (0 is also defensible); we use median.

Both are filled here for the persisted dataset; the modeling notebook will compute the
median on train only to avoid leakage.

In [4]:
df["income_missing"] = df["MonthlyIncome"].isnull().astype(int)
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())

df["NumberOfDependents"] = df["NumberOfDependents"].fillna(df["NumberOfDependents"].median())

print("Remaining missing values:", int(df.isnull().sum().sum()))
df[["MonthlyIncome", "NumberOfDependents"]].describe().T[["min", "max", "mean"]]

Remaining missing values: 0


,min,max,mean
MonthlyIncome,0.0000,"3,008,750.0000","6,418.4549"
NumberOfDependents,0.0000,20.0000,0.7374


## 4. Winsorize extreme outliers

`RevolvingUtilizationOfUnsecuredLines` and `DebtRatio` contain absurd values (a utilization
of 50,000, a debt ratio of 300,000). These are data errors, not signal. We **winsorize**
(clip) at the 99th percentile so extreme points can't dominate distance-based terms,
while keeping their rank information.

We also experienced `DebtRatio` chaos: some values are tiny when a borrower has high income.
Standard practice: clip at the 1st/99th percentiles for the two columns.

In [5]:
def winsorize(s, lower=0.01, upper=0.99):
    lo, hi = s.quantile(lower), s.quantile(upper)
    return s.clip(lo, hi)

before = df[["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]].describe().loc[["min", "max"]]
for c in ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]:
    df[c] = winsorize(df[c])
after = df[["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]].describe().loc[["min", "max"]]

print("BEFORE:"); print(before)
print("\nAFTER:"); print(after)

BEFORE:
     RevolvingUtilizationOfUnsecuredLines    DebtRatio
min                                0.0000       0.0000
max                           50,708.0000 329,664.0000

AFTER:
     RevolvingUtilizationOfUnsecuredLines  DebtRatio
min                                0.0000     0.0000
max                                1.0930 4,979.0400


## 5. Engineer credit-meaningful features

Raw columns are okay; domain features are better. We add a small, defensible set:

- **`total_past_due`** — sum of the three delinquency counts (overall delinquency burden).
- **`has_any_delinq`** — binary: any delinquency at all (step function the model can use).
- **`income_per_dependent`** — income divided by (dependents + 1), a crude affordability proxy.
- **`credit_lines_per_age`** — credit lines acquired per year of adulthood (credit appetite).
- **`is_high_util`** — utilization ≥ 0.9 (maxed-out flag).

We deliberately *avoid* stacking dozens of features; a few well-motivated ones beat
a kitchen sink and keep the model interpretable.

In [6]:
df["total_past_due"] = df[delinq_cols].sum(axis=1)
df["has_any_delinq"] = (df["total_past_due"] > 0).astype(int)
df["income_per_dependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)
df["credit_lines_per_age"] = df["NumberOfOpenCreditLinesAndLoans"] / df["age"].replace(0, np.nan)
df["credit_lines_per_age"] = df["credit_lines_per_age"].fillna(0)
df["is_high_util"] = (df["RevolvingUtilizationOfUnsecuredLines"] >= 0.9).astype(int)

print("Shape after FE:", df.shape)
new_cols = ["total_past_due", "has_any_delinq", "income_per_dependent", "credit_lines_per_age", "is_high_util"]
df[new_cols].describe().T

Shape after FE: (150000, 18)


,count,mean,std,min,25%,50%,75%,max
total_past_due,"150,000.0000",0.4004,1.1013,0.0000,0.0000,0.0000,0.0000,19.0000
has_any_delinq,"150,000.0000",0.2006,0.4005,0.0000,0.0000,0.0000,0.0000,1.0000
income_per_dependent,"150,000.0000","4,589.1192","7,974.4931",0.0000,"2,151.0000","4,000.0000","5,400.0000","1,794,060.0000"
credit_lines_per_age,"150,000.0000",0.1692,0.1052,0.0000,0.0938,0.1515,0.2250,1.3571
is_high_util,"150,000.0000",0.1338,0.3404,0.0000,0.0000,0.0000,0.0000,1.0000


## 6. Sanity checks & final feature list

- Confirm no missing values remain.
- Confirm no infinite values crept in from division.
- Separate target from features.

In [7]:
df = df.replace([np.inf, -np.inf], np.nan)

feature_cols = [c for c in df.columns if c != target]
assert df[feature_cols].isnull().sum().sum() == 0, "Unexpected NaNs remain!"
assert np.isfinite(df[feature_cols].to_numpy()).all(), "Non-finite values remain!"

print("Final feature count:", len(feature_cols))
print(feature_cols)

Final feature count: 17
['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'had_sentinel_delinq', 'income_missing', 'total_past_due', 'has_any_delinq', 'income_per_dependent', 'credit_lines_per_age', 'is_high_util']


## 7. Train / test split and persist

We hold out **20%** as a test set the model never sees. Stratify on the target so the
~6.7% default rate is preserved in both sets. The modeling notebook trains on `train`
and reports on `test`.

In [8]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df, test_size=0.20, random_state=RANDOM_STATE, stratify=df[target]
)

train.to_csv(f"{OUT_DIR}/train_clean.csv")
test.to_csv(f"{OUT_DIR}/test_clean.csv")

print(f"train: {train.shape[0]:,} rows | default rate {train[target].mean():.2%}")
print(f"test : {test.shape[0]:,} rows | default rate {test[target].mean():.2%}")
print("saved -> data/processed/train_clean.csv, test_clean.csv")

train: 120,000 rows | default rate 6.68%
test : 30,000 rows | default rate 6.68%
saved -> data/processed/train_clean.csv, test_clean.csv


### Takeaway

We now have a **cleaned, feature-engineered, leakage-safe split**:

| Handling | Decision |
|---|---|
| `age == 0` | → median |
| 96/98 sentinels | → missing + flag, impute 0 |
| `MonthlyIncome` missing | → flag + median |
| `NumberOfDependents` missing | → median |
| Utilization / DebtRatio outliers | → winsorized 1–99% |
| New features | 5 domain-derived |
| Split | 80/20 stratified |

Next: **04 — Modeling** (logistic baseline vs. XGBoost, class weights, cross-validation).